# 02 – Kat-İçi Ön İşleme, Manuel/Otomatik Arama ve 3×3 Nested CV
Bu defter **yalnızca Development** verisini kullanır (`data/dev_data.csv`). Test dosyası okunmaz.

Akış (Bölüm 5 ve 11):
1. Adayların nested CV başlamadan **önce** sabitlenip kaydedilmesi (M1–M6, R1–R6)
2. Outer 3-kat → her outer-train içinde inner 3-kat × 12 aday = **108 iç eğitim**
3. Her dış katta en iyi manuel ve en iyi otomatik adayın outer-train'de refit'i, outer-eval'de **bir kez** ölçüm (**6 refit**) + Dummy/LogReg referansları
4. Tam Development üzerinde 3-kat inner CV × 12 aday = **36 eğitim** → nihai konfigürasyon (`final_hyperparameters.json`)
5. İkincil eşik analizi (yalnızca development out-of-fold tahminleri)
6. Tablo 2–5, optimizer süre tablosu, uyarı kayıtları ve Şekil 5

Tüm bölünmeler kopya grubuna duyarlıdır ve her eğitim önbelleğe alınır (bağlantı koparsa kaldığı yerden devam eder).

Her eğitimin epoch bazlı train/early-stop kaybı ve LR değeri `results/training_logs_*.csv` dosyalarına yazılır (Bölüm 8).

In [ ]:
# === 0. Ortam kurulumu (üç defterde aynı) ===
import os, sys, subprocess, importlib

# Kendi GitHub reponuzun adresi (boş bırakılırsa Drive'daki YSA_Midterm_Project klasörü kullanılır)
GITHUB_REPO_URL = ""   # ör. "https://github.com/<kullanici-adi>/ysa-mlp-bank-marketing-suleyman-balsak.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if GITHUB_REPO_URL:
        repo = GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
        PROJECT_DIR = f"/content/drive/MyDrive/{repo}"
        if not os.path.exists(os.path.join(PROJECT_DIR, "src")):
            subprocess.run(["git", "clone", GITHUB_REPO_URL, PROJECT_DIR], check=True)
        else:   # yalnızca kod güncellemelerini çeker; üretilen sonuçlara dokunmaz
            r = subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], capture_output=True, text=True)
            print("git pull:", (r.stdout or r.stderr).strip()[:200])
    else:
        PROJECT_DIR = "/content/drive/MyDrive/YSA_Midterm_Project"
else:  # yerel Jupyter: defter proje kök dizininde çalıştırılır
    PROJECT_DIR = os.getcwd()

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for d in ["configs", "results", "results/cache", "figures", "data", "models"]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists("src/ysa_utils.py"):
    raise FileNotFoundError(
        f"{PROJECT_DIR}/src/ysa_utils.py bulunamadı!\n"
        "Çözüm 1: GITHUB_REPO_URL'ye kendi repo adresinizi yazın (repo src/ klasörünü içermeli).\n"
        f"Çözüm 2: Drive'da {PROJECT_DIR}/src/ klasörü açıp ysa_utils.py dosyasını oraya yükleyin.")

import src.ysa_utils as U
importlib.reload(U)                      # modül değişirse yeniden yükle
from src.ysa_utils import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Çalışma dizini:", os.getcwd())

In [ ]:
import json, time
import torch
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

SEEDS = load_json("configs/seeds.json")
dev_df = pd.read_csv("data/dev_data.csv")
X_dev = prepare_features(dev_df)            # kural tabanlı, fit yok
y_dev = dev_df[TARGET].astype(int)
g_dev = dev_df[GROUP_COL].values            # kopya grupları → grup-duyarlı bölünmeler
num_cols, cat_cols = get_feature_lists(X_dev)
assert TARGET not in X_dev.columns and ID_COL not in X_dev.columns
print(f"Development: {X_dev.shape} | pozitif oranı = {y_dev.mean():.4f} | "
      f"kopya grubu (≥2 satır): {int((pd.Series(g_dev).value_counts() > 1).sum())}")
print("Sayısal:", num_cols)
print("Kategorik:", cat_cols)
print("Seed'ler:", {k: v for k, v in SEEDS.items() if k != 'derivation'})
torch.set_num_threads(1)   # deterministik sonuçlar için tek thread

## 1. Hiperparametre adaylarının sabitlenmesi (nested CV'den ÖNCE)

In [ ]:
RANDOM_CONFIGS = make_random_configs(SEEDS["random_search_seed"])
ALL_CONFIGS = MANUAL_CONFIGS + RANDOM_CONFIGS
CFG = {c["config_id"]: c for c in ALL_CONFIGS}

assert len(MANUAL_CONFIGS) == 6 and len(RANDOM_CONFIGS) == 6
assert pd.Series([c["optimizer"] for c in RANDOM_CONFIGS]).value_counts().eq(2).all(), "Her optimizer için tam 2 aday olmalı"

save_json(MANUAL_CONFIGS, "configs/manual_candidates.json")
save_json(RANDOM_CONFIGS, "configs/random_candidates.json")
save_json(SEARCH_SPACE, "configs/search_space.json")
save_json(TRAINING_SETTINGS, "configs/training_settings.json")

cand_tbl = pd.DataFrame(ALL_CONFIGS)[["config_id", "search_type", "optimizer", "hidden_layers", "activation",
                                      "lr", "weight_decay", "batch_size", "scaler", "dropout", "use_scheduler",
                                      "clip_outliers", "unknown_strategy", "purpose"]]
cand_tbl.to_csv("configs/all_candidates_table.csv", index=False)
cand_tbl

> Bu noktada `configs/` klasörünü GitHub'a commit'lemeniz önerilir: adayların dış CV sonuçlarından önce sabitlendiğinin zaman damgalı kanıtı olur.

## 2. 3×3 Nested Cross-Validation (108 iç eğitim + 6 refit + referanslar)
- Katlar kopya grubuna duyarlıdır (`StratifiedGroupKFold`); aynı kopya hiçbir zaman hem eğitim hem değerlendirme tarafında olmaz.
- Her eğitim `results/cache/` altına kaydedilir. **Colab bağlantısı koparsa hücreyi yeniden çalıştırın, kaldığı yerden devam eder.** Konfigürasyon, seed veya bölünme değişirse önbellek otomatik geçersiz olur.

In [ ]:
def run_mlp(key, X_tr, y_tr, g_tr, X_ev, cfg, model_seed, es_seed):
    """Önbellekli tek MLP eğitimi → JSON kayıt (olasılıklar, geçmiş, bilgi)."""
    extra = {"model_seed": model_seed, "es_seed": es_seed,
             "train_idx": index_hash(X_tr.index), "eval_idx": index_hash(X_ev.index)}
    def fn():
        res = train_mlp(X_tr, y_tr, {"eval": X_ev}, cfg, num_cols, cat_cols,
                        model_seed=model_seed, es_seed=es_seed, groups=g_tr)
        return result_record(res, "eval")
    return run_cached(key, cfg, extra, fn)


def fit_logreg(X_tr, y_tr, X_ev):
    """Referans: sabit ön işleme kuralı (Standard, unknown ayrı kategori, kırpma yok)."""
    t0 = time.time()
    pre = build_preprocessor(num_cols, cat_cols, "StandardScaler", False, "keep_category")
    lr = LogisticRegression(max_iter=2000, random_state=SEEDS["logreg_seed"])
    lr.fit(pre.fit_transform(X_tr), y_tr)
    p = lr.predict_proba(pre.transform(X_ev))[:, 1]
    return p, int(lr.coef_.size + lr.intercept_.size), time.time() - t0


outer_splits = make_cv_splits(y_dev, g_dev, 3, SEEDS["outer_cv_seed"])
inner_rows, outer_rows, selection_rows, log_frames, fold_rows = [], [], [], [], []
T_START = time.time()
n_cached = 0

for k, (otr, oev) in enumerate(outer_splits, start=1):
    X_otr, y_otr, g_otr = X_dev.iloc[otr], y_dev.iloc[otr], g_dev[otr]
    X_oev, y_oev = X_dev.iloc[oev], y_dev.iloc[oev]
    assert not set(g_otr) & set(g_dev[oev]), "Kopya grubu dış katlar arasında bölünmüş!"
    fold_rows.append({"outer_fold": k, "n_outer_train": len(otr), "n_outer_eval": len(oev),
                      "pos_outer_train": int(y_otr.sum()), "pos_outer_eval": int(y_oev.sum()),
                      "pos_rate_outer_eval": round(float(y_oev.mean()), 4)})

    inner_seed = SEEDS["inner_cv_seed"] + k
    inner_splits = make_cv_splits(y_otr, g_otr, 3, inner_seed)

    # --- 2a. İç CV: 12 aday × 3 iç kat (tüm adaylar AYNI iç bölünmeleri kullanır) ---
    for cfg in ALL_CONFIGS:
        for j, (itr, iva) in enumerate(inner_splits, start=1):
            res = run_mlp(f"outer{k}_inner{j}_{cfg['config_id']}",
                          X_otr.iloc[itr], y_otr.iloc[itr], g_otr[itr], X_otr.iloc[iva], cfg,
                          model_seed=SEEDS["model_init_seeds"][0] + 100 * k + j,
                          es_seed=SEEDS["early_stop_seed"] + 100 * k + j)
            n_cached += res["from_cache"]
            inner_rows.append({
                "outer_fold": k, "inner_fold": j, "inner_cv_seed": inner_seed,
                "config_id": cfg["config_id"], "search_type": cfg["search_type"], "optimizer": cfg["optimizer"],
                "inner_AP": float(average_precision_score(y_otr.iloc[iva], res["probs"])),
                "best_epoch": res["best_epoch"], "stopped_epoch": res["stopped_epoch"],
                "n_lr_drops": res["n_lr_drops"], "final_lr": res["final_lr"],
                "best_es_loss": res["best_es_loss"], "n_params": res["n_params"], "input_dim": res["input_dim"],
                "n_fit": res["n_fit"], "n_es": res["n_es"], "n_es_pos": res["n_es_pos"],
                "elapsed_sec": res["elapsed_sec"], "status": res["status"], "warnings": res["warnings"],
            })
            log_frames.append(history_frame(res, phase="nested_inner", outer_fold=k, inner_fold=j,
                                            config_id=cfg["config_id"]))

    # --- 2b. Bu dış kat için seçim: iç CV ortalama AP (eşitlikte config_id sırası) ---
    df_k = pd.DataFrame([r for r in inner_rows if r["outer_fold"] == k])
    summ_k = (df_k.groupby(["search_type", "config_id"])
              .agg(mean=("inner_AP", "mean"), std=("inner_AP", "std")).reset_index())
    for method, stype in [("MLP_Manual", "manual"), ("MLP_Random", "random")]:
        sub = summ_k[summ_k["search_type"] == stype].sort_values(["mean", "config_id"], ascending=[False, True])
        sel_id = sub.iloc[0]["config_id"]
        selection_rows.append({"outer_fold": k, "method": method, "selected_config": sel_id,
                               "inner_mean_AP": float(sub.iloc[0]["mean"]), "inner_sd_AP": float(sub.iloc[0]["std"])})

        # --- 2c. Outer-train üzerinde refit → outer-eval'de BİR KEZ ölçüm ---
        res = run_mlp(f"outer{k}_refit_{method}_{sel_id}", X_otr, y_otr, g_otr, X_oev, CFG[sel_id],
                      model_seed=SEEDS["model_init_seeds"][0] + 1000 * k,
                      es_seed=SEEDS["early_stop_seed"] + 1000 * k)
        m = compute_metrics(y_oev, res["probs"])
        m.update({"outer_fold": k, "method": method, "selected_config": sel_id,
                  "best_epoch": res["best_epoch"], "n_params": res["n_params"],
                  "elapsed_sec": res["elapsed_sec"], "status": res["status"]})
        outer_rows.append(m)
        log_frames.append(history_frame(res, phase="outer_refit", outer_fold=k, inner_fold=0, config_id=sel_id))

    # --- 2d. Referans modeller (outer-train'de eğitilir, outer-eval'de ölçülür) ---
    t0 = time.time()
    dummy = DummyClassifier(strategy="most_frequent").fit(X_otr, y_otr)
    p_dummy = dummy.predict_proba(X_oev)[:, 1]
    m = compute_metrics(y_oev, p_dummy)
    m.update({"outer_fold": k, "method": "Dummy_MostFrequent", "selected_config": "most_frequent",
              "best_epoch": np.nan, "n_params": 0, "elapsed_sec": time.time() - t0, "status": "ok"})
    outer_rows.append(m)

    p_lr, n_lr, t_lr = fit_logreg(X_otr, y_otr, X_oev)
    m = compute_metrics(y_oev, p_lr)
    m.update({"outer_fold": k, "method": "Logistic_Regression", "selected_config": "L2, C=1.0",
              "best_epoch": np.nan, "n_params": n_lr, "elapsed_sec": t_lr, "status": "ok"})
    outer_rows.append(m)

    sel_txt = ", ".join(f"{r['method']}→{r['selected_config']}" for r in selection_rows if r["outer_fold"] == k)
    print(f"Dış kat {k} tamam ({time.time() - T_START:.0f} sn) | seçimler: {sel_txt}")

inner_df = pd.DataFrame(inner_rows)
outer_df = pd.DataFrame(outer_rows)
selection_df = pd.DataFrame(selection_rows)
assert len(inner_df) == 3 * 3 * 12, len(inner_df)

inner_df.to_csv("results/nested_inner_cv_raw.csv", index=False)
outer_df.to_csv("results/outer_cv_results.csv", index=False)
selection_df.to_csv("results/outer_fold_selections.csv", index=False)
pd.DataFrame(fold_rows).to_csv("results/outer_fold_sizes.csv", index=False)
pd.concat(log_frames, ignore_index=True).to_csv("results/training_logs_nested_cv.csv", index=False)
print(f"\nToplam süre: {time.time() - T_START:.0f} sn | iç eğitim: {len(inner_df)} | dış refit: {len(selection_df)} "
      f"| önbellekten gelen iç eğitim: {n_cached}")

### Dış katlardaki iç CV özetleri (her dış kat × 12 aday)

In [ ]:
inner_summary = (inner_df.groupby(["outer_fold", "search_type", "config_id"])
                 .agg(mean_AP=("inner_AP", "mean"), sd_AP=("inner_AP", "std"),
                      mean_best_epoch=("best_epoch", "mean"), total_time_sec=("elapsed_sec", "sum"))
                 .reset_index())
inner_summary.to_csv("results/nested_inner_cv_summary.csv", index=False)

pivot = inner_summary.pivot(index="config_id", columns="outer_fold", values="mean_AP").round(4)
pivot.columns = [f"Dış kat {c} – iç mean AP" for c in pivot.columns]
display(pivot)
display(selection_df)

### Tablo 4 – Dış kat bazlı ham sonuçlar ve Tablo 5 – Mean ± SD (n = 3)

In [ ]:
show_cols = ["outer_fold", "method", "selected_config", "AP", "ROC_AUC", "F1", "Sensitivity", "Specificity",
             "Balanced_Accuracy", "Accuracy", "Precision", "TP", "FP", "TN", "FN", "elapsed_sec"]
display(outer_df[show_cols].round(4))

outer_summary = mean_sd_table(outer_df, "method")
outer_summary.to_csv("results/outer_cv_summary_mean_sd.csv", index=False)
display(outer_summary[["method", "n"] + METRIC_COLS])
print("Not: 3 dış kat bağımsız tekrarlar değildir (eğitim kümeleri örtüşür); SD bir güven aralığı değildir.")

## 3. Tam Development üzerinde nihai seçim (3 iç kat × 12 aday = 36 eğitim)

In [ ]:
full_seed = SEEDS["inner_cv_seed"]
dev_splits = make_cv_splits(y_dev, g_dev, 3, full_seed)
full_rows, full_logs = [], []
oof = {c["config_id"]: np.full(len(y_dev), np.nan) for c in ALL_CONFIGS}   # out-of-fold olasılıklar
T0 = time.time()
for cfg in ALL_CONFIGS:
    for j, (itr, iva) in enumerate(dev_splits, start=1):
        res = run_mlp(f"fulldev_inner{j}_{cfg['config_id']}", X_dev.iloc[itr], y_dev.iloc[itr], g_dev[itr],
                      X_dev.iloc[iva], cfg,
                      model_seed=SEEDS["model_init_seeds"][0] + 5000 + j,
                      es_seed=SEEDS["early_stop_seed"] + 5000 + j)
        oof[cfg["config_id"]][iva] = res["probs"]
        full_rows.append({
            "config_id": cfg["config_id"], "search_type": cfg["search_type"], "inner_fold": j,
            "inner_AP": float(average_precision_score(y_dev.iloc[iva], res["probs"])),
            "best_epoch": res["best_epoch"], "stopped_epoch": res["stopped_epoch"],
            "n_lr_drops": res["n_lr_drops"], "n_params": res["n_params"], "input_dim": res["input_dim"],
            "elapsed_sec": res["elapsed_sec"], "status": res["status"], "warnings": res["warnings"],
        })
        full_logs.append(history_frame(res, phase="full_dev_inner", outer_fold=0, inner_fold=j,
                                       config_id=cfg["config_id"]))
full_dev_df = pd.DataFrame(full_rows)
assert len(full_dev_df) == 36
full_dev_df.to_csv("results/full_dev_inner_cv_raw.csv", index=False)
oof_df = pd.DataFrame({ID_COL: dev_df[ID_COL].values, TARGET: y_dev.values, **{f"oof_{c}": v for c, v in oof.items()}})
assert not oof_df.isna().any().any()
oof_df.to_csv("results/full_dev_oof_predictions.csv", index=False)
pd.concat(full_logs, ignore_index=True).to_csv("results/training_logs_full_dev.csv", index=False)
print(f"36 eğitim tamamlandı: {time.time() - T0:.0f} sn")

# ÖNEMLİ: gruplama YALNIZCA config_id ile yapılır. n_params fold'a göre değişebilir
# (one-hot kategori sayısı), bu yüzden anahtar değil, özet istatistiktir.
agg = (full_dev_df.groupby("config_id")
       .agg(mean_AP=("inner_AP", "mean"), sd_AP=("inner_AP", "std"), n_folds=("inner_AP", "size"),
            mean_best_epoch=("best_epoch", "mean"), mean_stopped_epoch=("stopped_epoch", "mean"),
            mean_lr_drops=("n_lr_drops", "mean"), n_params=("n_params", "median"),
            mean_time_per_fit_sec=("elapsed_sec", "mean"), total_time_sec=("elapsed_sec", "sum"))
       .reset_index())
assert (agg["n_folds"] == 3).all()
params = pd.DataFrame(ALL_CONFIGS).drop(columns=["purpose"])
params["hidden_layers"] = params["hidden_layers"].astype(str)
search_summary = params.merge(agg, on="config_id").sort_values("mean_AP", ascending=False)

tbl_cols = ["config_id", "optimizer", "hidden_layers", "activation", "lr", "weight_decay", "batch_size",
            "scaler", "dropout", "use_scheduler", "clip_outliers", "unknown_strategy", "n_params",
            "mean_AP", "sd_AP", "mean_best_epoch", "mean_lr_drops", "total_time_sec"]
manual_tbl = search_summary[search_summary.search_type == "manual"][tbl_cols]
random_tbl = search_summary[search_summary.search_type == "random"][tbl_cols]
manual_tbl.to_csv("results/manual_search_results.csv", index=False)
random_tbl.to_csv("results/random_search_results.csv", index=False)
print("Tablo 2 – Manuel arama (tam dev iç CV):"); display(manual_tbl.round(4))
print("Tablo 3 – Otomatik arama (tam dev iç CV):"); display(random_tbl.round(4))

### Önceden tanımlanmış seçim kuralı
1. Manuel aramanın en iyisi = M1–M6 içinde en yüksek tam-dev iç CV mean AP (eşitlikte düşük SD)
2. Otomatik aramanın en iyisi = R1–R6 içinde en yüksek tam-dev iç CV mean AP
3. Nihai yöntem = bu ikisinden mean AP'si yüksek olan (eşitlikte düşük SD, sonra manuel)

Dış CV sonuçları bu seçimde **kullanılmaz**.

**İkincil eşik analizi (opsiyonel, Bölüm 8):** Birincil sınıflandırma eşiği 0.50'dir. 0.50'de duyarlılık düşük kaldığı için ikincil bir eşik de belirlenir: nihai konfigürasyonun tam-dev iç CV *out-of-fold* tahminlerinde F1'i en yüksek yapan eşik. Bu eşik yalnızca development verisiyle seçilir, test tahminleri kilitlenmeden önce kaydedilir ve testte **ikincil sonuç** olarak raporlanır. Ana sonuç 0.50 eşiğidir.

In [ ]:
def best_of(stype):
    s = search_summary[search_summary.search_type == stype].sort_values(["mean_AP", "sd_AP"], ascending=[False, True])
    return s.iloc[0]

bm, br = best_of("manual"), best_of("random")
if (br["mean_AP"] > bm["mean_AP"]) or (br["mean_AP"] == bm["mean_AP"] and br["sd_AP"] < bm["sd_AP"]):
    win, method = br, "random_search"
else:
    win, method = bm, "manual_search"

FINAL_CFG = dict(CFG[win["config_id"]])

# İkincil (opsiyonel) eşik analizi – Bölüm 8 ve 12: eşik YALNIZCA development iç CV
# (out-of-fold) tahminleriyle seçilir. Birincil sonuç her zaman 0.50 eşiğidir.
oof_lr = np.full(len(y_dev), np.nan)
for itr, iva in dev_splits:
    oof_lr[iva], _, _ = fit_logreg(X_dev.iloc[itr], y_dev.iloc[itr], X_dev.iloc[iva])
thr_mlp = f1_optimal_threshold(y_dev, oof[win["config_id"]])
thr_lr = f1_optimal_threshold(y_dev, oof_lr)
thr_tbl = pd.DataFrame([
    {"Model": f"MLP {win['config_id']}", "Eşik": t, **{m: compute_metrics(y_dev, p, t)[m] for m in
     ["F1", "Sensitivity", "Specificity", "Precision", "Balanced_Accuracy"]}}
    for t, p in [(0.50, oof[win["config_id"]]), (thr_mlp, oof[win["config_id"]])]] + [
    {"Model": "Logistic Regression", "Eşik": t, **{m: compute_metrics(y_dev, oof_lr, t)[m] for m in
     ["F1", "Sensitivity", "Specificity", "Precision", "Balanced_Accuracy"]}} for t in (0.50, thr_lr)])
thr_tbl.round(4).to_csv("results/threshold_selection_dev_oof.csv", index=False)

FINAL_CFG.update({
    "selected_search_method": method,
    "selection_rule": "tam development 3-kat iç CV mean AP (eşitlikte düşük SD, sonra manuel); dış CV kullanılmadı",
    "full_dev_mean_AP": float(win["mean_AP"]), "full_dev_sd_AP": float(win["sd_AP"]),
    "best_manual": {"config_id": bm["config_id"], "mean_AP": float(bm["mean_AP"]), "sd_AP": float(bm["sd_AP"])},
    "best_random": {"config_id": br["config_id"], "mean_AP": float(br["mean_AP"]), "sd_AP": float(br["sd_AP"])},
    "threshold": TRAINING_SETTINGS["threshold"],
    "secondary_threshold_mlp": thr_mlp, "secondary_threshold_logreg": thr_lr,
    "secondary_threshold_rule": "tam development 3-kat iç CV out-of-fold tahminlerinde F1-optimal eşik (ikincil analiz)",
    "full_dev_inner_cv_seed": full_seed,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
})
save_json(FINAL_CFG, "results/final_hyperparameters.json")
print(f"En iyi manuel : {bm['config_id']}  AP = {bm['mean_AP']:.4f} ± {bm['sd_AP']:.4f}")
print(f"En iyi otomatik: {br['config_id']}  AP = {br['mean_AP']:.4f} ± {br['sd_AP']:.4f}")
print(f"\n>>> NİHAİ: {FINAL_CFG['config_id']} ({method}) – {config_label(FINAL_CFG)}")
print(f"\nİkincil eşik analizi (development OOF): MLP F1-optimal eşik = {thr_mlp:.3f} | LogReg = {thr_lr:.3f}")
display(thr_tbl.round(4))

## 4. Optimizer süreleri, yakınsama ve uyarı kayıtları (Bölüm 10.4, 17.6)

In [ ]:
all_runs = pd.concat([inner_df.assign(phase="nested_inner"),
                      full_dev_df.assign(phase="full_dev_inner")], ignore_index=True)
all_runs["optimizer"] = all_runs["config_id"].map(lambda c: CFG[c]["optimizer"])

opt_tbl = (all_runs.groupby("optimizer")
           .agg(n_trainings=("config_id", "size"), mean_time_sec=("elapsed_sec", "mean"),
                total_time_sec=("elapsed_sec", "sum"), mean_best_epoch=("best_epoch", "mean"),
                mean_stopped_epoch=("stopped_epoch", "mean"), mean_lr_drops=("n_lr_drops", "mean"))
           .round(3).reset_index())
opt_tbl.to_csv("results/optimizer_time_summary.csv", index=False)
display(opt_tbl)

status_tbl = all_runs.groupby(["config_id", "status"]).size().unstack(fill_value=0)
status_tbl.to_csv("results/training_status_summary.csv")
display(status_tbl)

warn = all_runs[all_runs["warnings"].fillna("") != ""][["phase", "config_id", "outer_fold", "inner_fold",
                                                         "best_epoch", "status", "warnings"]]
warn.to_csv("results/training_warnings.csv", index=False)
print(f"Uyarılı eğitim sayısı: {len(warn)} / {len(all_runs)}")
display(warn.groupby("config_id").size().rename("uyarı sayısı").to_frame().T)

# M1 vs M6: scheduler gerçekten LR düşürdü mü? (H6)
print("\nM1 (scheduler açık) LR düşüş sayısı (ort.):",
      round(all_runs.loc[all_runs.config_id == "M1", "n_lr_drops"].mean(), 2),
      "| M1 ve M6 iç AP farkı (tam dev):",
      round(search_summary.set_index("config_id").loc["M1", "mean_AP"]
            - search_summary.set_index("config_id").loc["M6", "mean_AP"], 6))

## 5. Şekil 5 – Manuel ve otomatik arama karşılaştırması

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.2, 1]})

# 5a: tam dev iç CV mean AP ± SD
d = search_summary.sort_values("mean_AP")
cols = ["#4C72B0" if t == "manual" else "#DD8452" for t in d["search_type"]]
axes[0].barh(d["config_id"], d["mean_AP"], xerr=d["sd_AP"], color=cols, edgecolor="black", capsize=4)
for i, (v, sd) in enumerate(zip(d["mean_AP"], d["sd_AP"].fillna(0))):
    axes[0].text(v + sd + 0.004, i, f"{v:.3f}", va="center", fontsize=8)
axes[0].axvline(y_dev.mean(), color="gray", ls=":", label=f"Pozitif oran (rastgele AP) = {y_dev.mean():.3f}")
axes[0].set_xlabel("Tam development iç CV AP (Mean ± SD, n = 3)")
axes[0].set_title("(a) 12 aday – iç CV AP")
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color="#4C72B0", label="Manuel (M1–M6)"),
                        Patch(color="#DD8452", label="Otomatik (R1–R6)")] +
               axes[0].get_legend_handles_labels()[0], loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
axes[0].set_xlim(0, (d["mean_AP"] + d["sd_AP"].fillna(0)).max() * 1.2)

# 5b: dış CV – yöntemler, kat bazlı noktalar + ortalama
order = ["Dummy_MostFrequent", "Logistic_Regression", "MLP_Manual", "MLP_Random"]
for mi, met in enumerate(["AP", "ROC_AUC"]):
    for i, mth in enumerate(order):
        v = outer_df.loc[outer_df.method == mth, met].values
        x = i + (mi - 0.5) * 0.3
        axes[1].scatter([x] * len(v), v, color=["#55A868", "#C44E52"][mi], alpha=0.6, s=25)
        axes[1].errorbar(x, v.mean(), yerr=v.std(ddof=1), fmt="D", color=["#55A868", "#C44E52"][mi],
                         capsize=4, label=met if i == 0 else None)
axes[1].set_xticks(range(len(order)))
axes[1].set_xticklabels(["Dummy", "LogReg", "MLP\nManuel", "MLP\nOtomatik"])
axes[1].set_ylabel("Dış kat skoru")
axes[1].set_title("(b) Dış CV: kat sonuçları ve Mean ± SD (n = 3)")
axes[1].legend()
plt.tight_layout(); plt.savefig("figures/fig5_manual_vs_random.png", dpi=300); plt.show()

## 6. Bu defterin ürettiği dosyalar

In [ ]:
for f in ["configs/manual_candidates.json", "configs/random_candidates.json", "configs/search_space.json",
          "configs/training_settings.json", "results/nested_inner_cv_raw.csv", "results/nested_inner_cv_summary.csv",
          "results/outer_fold_selections.csv", "results/outer_cv_results.csv", "results/outer_cv_summary_mean_sd.csv",
          "results/full_dev_inner_cv_raw.csv", "results/full_dev_oof_predictions.csv", "results/threshold_selection_dev_oof.csv", "results/manual_search_results.csv", "results/random_search_results.csv",
          "results/final_hyperparameters.json", "results/optimizer_time_summary.csv", "results/training_warnings.csv",
          "results/training_logs_nested_cv.csv", "results/training_logs_full_dev.csv", "figures/fig5_manual_vs_random.png"]:
    print(("✓ " if os.path.exists(f) else "✗ ") + f)